Vector database

In [1]:
!pip install chromadb

In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    faiss-cpu \
    google_genai

In [3]:
pip install langchain-google-genai

Dependencies

In [4]:
from google.colab import userdata
import os

In [5]:
os.environ['GOOGLE_API_KEY'] = userdata.get('gemini_key')

Installing the Embedding Model

In [21]:
#from langchain.embeddings import HuggingFaceEmbeddings
#embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#vectordb = Chroma.from_documents(docs, embedding=embedding)

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

google_embd_model = GoogleGenerativeAIEmbeddings(
                                                 model="models/gemini-embedding-001")

In [7]:
# Embed single query

embedded_query = google_embd_model.embed_query("What was the name mentioned in the conversation?")

print("Dimensionality of embedded vector:", len(embedded_query))
print()
print(embedded_query[:15])


Dimensionality of embedded vector: 3072

[-0.030619625, 0.0032838325, 0.015690394, -0.0870063, -0.008714505, 0.004057516, -0.011523551, 0.00695804, -0.022802716, 0.010272241, -0.0064204945, -0.007641482, -0.0015980989, 0.007948611, 0.11037036]


Embed list of Documents using `embed_documents()`

In [8]:
docs = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

In [9]:
embeddings = google_embd_model.embed_documents(docs)

print("Number of Embeddings:",len(embeddings))
print("Demension of Embeddings:", len(embeddings[0]))


Number of Embeddings: 5
Demension of Embeddings: 3072


Chromadb

In [10]:
import chromadb

client = chromadb.PersistentClient(path="vector_store")

In [11]:
from datetime import datetime

collection = client.create_collection(
    name="my_first_collection",
    embedding_function=None,
    metadata={
        "description": "my first Chroma collection",
        "created": str(datetime.now()),
        "hnsw:space": "cosine"
    }
)

In [13]:
collection.count()

0

In [15]:
collection.peek()

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

In [16]:
import uuid

# add embeddings and documents
data = [
    "Apples - High in fiber, support digestion, and promote heart health.",
    "Bananas - Rich in potassium, help regulate blood pressure and muscle function.",
    "Oranges - Packed with vitamin C, boost immunity, and promote skin health.",
    "Blueberries - High in antioxidants, improve brain function and reduce inflammation.",
    "Strawberries - Support heart health and contain anti-aging antioxidants.",
    "Watermelon - Hydrating fruit with lycopene, good for heart and skin health.",
    "Pineapple - Contains bromelain, aids digestion, and reduces inflammation.",
    "Avocado - Loaded with healthy fats, supports brain and heart health.",
    "Papaya - Rich in enzymes for digestion and boosts skin health.",
    "Pomegranate - Full of antioxidants, improves blood circulation, and heart health.",
    "Carrots - High in beta-carotene, improve eye health and skin glow.",
    "Spinach - Rich in iron, good for blood health and energy levels.",
    "Broccoli - Contains sulforaphane, which has anti-cancer properties.",
    "Tomatoes - Packed with lycopene, supports heart health and skin protection.",
    "Bell Peppers - High in vitamin C, boosts immunity, and reduces inflammation.",
    "Cucumber - Hydrating vegetable, aids in digestion, and supports skin health.",
    "Garlic - Has antibacterial properties, supports heart health and immunity.",
    "Ginger - Anti-inflammatory, helps with digestion and nausea relief.",
    "Beets - Improve blood flow, support endurance, and detox the liver.",
    "Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion."
    ]

collection.add(
    documents = data,
    ids=[ str(uuid.uuid4()) for i in range(len(data)) ]
    # metadatas=[{"key_1": "abc_1", "key_2": "abc_2"}, {"key_1": "xyz_1", "key_2": "xyz_2"}]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 51.2MiB/s]


In [17]:
collection.count()

20

In [18]:
collection.peek(1)

{'ids': ['9234635b-00b3-4967-97ba-dcec1e0e556c'],
 'embeddings': array([[ 4.32430953e-02,  7.91505910e-03,  2.22158618e-02,
          9.66520011e-02, -2.08716057e-02,  1.83702335e-02,
          3.16199660e-02, -3.56986523e-02,  3.43051436e-03,
         -3.65285985e-02,  5.40717430e-02, -1.16434470e-02,
         -1.14744566e-02, -5.66731617e-02,  4.13682237e-02,
         -3.38005759e-02,  5.06379306e-02, -5.43545559e-03,
         -6.15437441e-02, -2.64442936e-02, -1.59990378e-02,
          2.24989560e-02,  7.18898773e-02,  4.56595048e-02,
          1.90453674e-03, -1.01699587e-02,  2.42284890e-02,
         -2.40649693e-02, -8.08296204e-02, -1.55629432e-02,
          5.21274917e-02, -7.25148711e-03,  1.40228972e-01,
          5.08025102e-02, -5.28849177e-02,  1.30572179e-02,
          2.08559692e-01, -5.16417623e-02, -9.83181223e-02,
         -3.38158831e-02,  3.49974819e-03,  1.32071292e-02,
          4.49088402e-02,  7.41897076e-02,  2.94162519e-03,
          7.66488630e-03,  1.0677783

Search Embeddings

In [19]:
results = collection.query(
    query_texts=["digestion, fiber rich"],
    n_results=3
)

results

{'ids': [['d24f5f95-d239-40ed-b382-30a71901e7fa',
   '9234635b-00b3-4967-97ba-dcec1e0e556c',
   '8aa7f4fd-ef7e-4402-9305-784f1331b5c8']],
 'embeddings': None,
 'documents': [['Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.',
   'Apples - High in fiber, support digestion, and promote heart health.',
   'Spinach - Rich in iron, good for blood health and energy levels.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.4631820321083069, 0.49158740043640137, 0.5677580833435059]]}

In [20]:

results = collection.query(
    query_texts=["vitamin rich"],
    n_results=3
)

results["documents"]

[['Spinach - Rich in iron, good for blood health and energy levels.',
  'Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.',
  'Oranges - Packed with vitamin C, boost immunity, and promote skin health.']]